In [ ]:
import duckdb
import tomllib
from pathlib import Path

# set the dataset for gathering information about the budget, popularity and vote average of movies released in the last 10 years with a budget greater than $10,000, runtime greater than 40 minutes, and that are not adult films. We will also filter out movies with zero popularity or zero vote average to focus on more relevant data.

secrets = tomllib.loads(Path("/Users/patyrocks/Documents/pyground/ETL/.streamlit/secrets.toml").read_text())
token = secrets["MOTHERDUCK_TOKEN"]

con = duckdb.connect(f"md:TMDB?motherduck_token={token}", read_only=True)

budget_pop = con.sql("select title, budget, popularity, vote_average, vote_count from movies where budget > 10000  and popularity > 0  and vote_average > 0  AND adult = false and runtime>40 and release_date >= '2015-01-01' and release_date < '2025-01-01'")

df = budget_pop.df()

# Remove outliers on vote_count (1st-99th percentile)
q01_vc = df['vote_count'].quantile(0.01)
q99_vc = df['vote_count'].quantile(0.99)
df = df[(df['vote_count'] >= q01_vc) & (df['vote_count'] <= q99_vc)]

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

# make it into a bell curve distribution like
df['log_budget'] = np.log10(df['budget'])

fig = px.histogram(df, x="log_budget", nbins=50, title="Budget Distribution (Log Scale)",
                   labels={"log_budget": "Budget (log₁₀ USD)"})
fig.update_layout(
    yaxis_title="Number of Movies",
    xaxis=dict(
        tickvals=[4, 5, 6, 7, 8, 9],
        ticktext=["$10K", "$100K", "$1M", "$10M", "$100M", "$1B"]
    )
)
fig.show()

In [8]:
df_pop = df

df_pop['log_popularity'] = np.log10(df_pop['popularity'])

fig = px.histogram(df_pop, x="log_popularity", nbins=50, title="Popularity Distribution (Log Scale)",
                   labels={"log_popularity": "Popularity (log₁₀)"})
fig.update_layout(
    yaxis_title="Number of Movies",
    xaxis=dict(
        tickvals=[0, 0.5, 1, 1.5, 2, 2.5, 3],
        ticktext=["1", "3", "10", "30", "100", "300", "1K"]
    )
)
fig.show()

In [13]:
q01 = df['vote_average'].quantile(0.01)
q99 = df['vote_average'].quantile(0.99)
df_vote = df[(df['vote_average'] >= q01) & (df['vote_average'] <= q99)]

fig = px.histogram(df_vote, x="vote_average", nbins=50, title="Vote Average Distribution (1st-99th percentile)",
                   labels={"vote_average": "Vote Average"})
fig.update_layout(yaxis_title="Number of Movies")
fig.show()

In [14]:
import plotly.express as px
import numpy as np

# Query revenue data with same filters as budget
rev = con.sql("select title, revenue from movies where revenue > 10000 and budget > 10000 and popularity > 0 and vote_average > 0 AND adult = false and runtime > 40 and release_date >= '2015-01-01' and release_date < '2025-01-01'")
df_rev = rev.df()

# Remove outliers (1st-99th percentile)
q01 = df_rev['revenue'].quantile(0.01)
q99 = df_rev['revenue'].quantile(0.99)
df_rev = df_rev[(df_rev['revenue'] >= q01) & (df_rev['revenue'] <= q99)]

df_rev['log_revenue'] = np.log10(df_rev['revenue'])

fig = px.histogram(df_rev, x="log_revenue", nbins=50, title="Revenue Distribution (Log Scale)",
                   labels={"log_revenue": "Revenue (log₁₀ USD)"})
fig.update_layout(
    yaxis_title="Number of Movies",
    xaxis=dict(
        tickvals=[4, 5, 6, 7, 8, 9],
        ticktext=["$10K", "$100K", "$1M", "$10M", "$100M", "$1B"]
    )
)
fig.show()

In [15]:
import numpy as np
import plotly.graph_objects as go
import math

# Shared sample: same filters as dashboard (budget>0, revenue>0, vote_count>=100, 1st-99th percentile on all dims)
hm_df = con.sql("""
    select popularity, vote_average, budget, revenue
    from movies
    where budget > 0 and revenue > 0 and popularity > 0 and vote_average > 0 and vote_count >= 100
    and adult = false and EXTRACT(YEAR FROM release_date) <= EXTRACT(YEAR FROM current_date())
    and budget between
      (select percentile_cont(0.01) within group (order by budget) from movies where budget > 0 and adult = false)
      and (select percentile_cont(0.99) within group (order by budget) from movies where budget > 0 and adult = false)
    and revenue between
      (select percentile_cont(0.01) within group (order by revenue) from movies where revenue > 0 and adult = false)
      and (select percentile_cont(0.99) within group (order by revenue) from movies where revenue > 0 and adult = false)
    and popularity between
      (select percentile_cont(0.01) within group (order by popularity) from movies where popularity > 0 and adult = false)
      and (select percentile_cont(0.99) within group (order by popularity) from movies where popularity > 0 and adult = false)
    and vote_average between
      (select percentile_cont(0.01) within group (order by vote_average) from movies where vote_average > 0 and adult = false)
      and (select percentile_cont(0.99) within group (order by vote_average) from movies where vote_average > 0 and adult = false)
    and vote_count between
      (select percentile_cont(0.01) within group (order by vote_count) from movies where vote_count > 0 and adult = false)
      and (select percentile_cont(0.99) within group (order by vote_count) from movies where vote_count > 0 and adult = false)
""").fetchall()

n_bins = 8

# Quantile edges for popularity
pops = sorted([p for p, v, b, r in hm_df])
pop_edges = [pops[int(len(pops) * i / n_bins)] for i in range(n_bins)] + [float('inf')]

# Quantile edges for vote average
votes = sorted([v for p, v, b, r in hm_df])
vote_edges = [votes[int(len(votes) * i / n_bins)] for i in range(n_bins)] + [float('inf')]

# Quantile edges for budget
budgets = sorted([b for p, v, b, r in hm_df])
budget_edges = [budgets[int(len(budgets) * i / n_bins)] for i in range(n_bins)] + [float('inf')]

def fmt_edge(v):
    return f'{v:.0f}' if v >= 10 else f'{v:.1f}'

def fmt_budget(v):
    if v >= 1_000_000: return f'${v/1_000_000:.0f}M'
    if v >= 1_000: return f'${v/1_000:.0f}K'
    return f'${v:.0f}'

pop_labels = [f'{fmt_edge(pop_edges[i])}-{fmt_edge(pop_edges[i+1])}' if pop_edges[i+1] != float('inf') else f'{fmt_edge(pop_edges[i])}+' for i in range(n_bins)]
vote_labels = [f'{vote_edges[i]:.1f}-{vote_edges[i+1]:.1f}' if vote_edges[i+1] != float('inf') else f'{vote_edges[i]:.1f}+' for i in range(n_bins)]
budget_labels = [f'{fmt_budget(budget_edges[i])}-{fmt_budget(budget_edges[i+1])}' if budget_edges[i+1] != float('inf') else f'{fmt_budget(budget_edges[i])}+' for i in range(n_bins)]

def find_bin(val, edges):
    return next((i for i in range(len(edges) - 1) if edges[i] <= val < edges[i + 1]), n_bins - 1)

print(f"Shared sample: {len(hm_df):,} movies")

Shared sample: 4,007 movies


In [16]:
# Heatmap 1: Popularity (X) × Vote Average (Y) → Avg log₁₀(Budget) color
hm1 = {}
for pop, vote, budget, revenue in hm_df:
    pi, vi = find_bin(pop, pop_edges), find_bin(vote, vote_edges)
    key = (vi, pi)
    if key not in hm1:
        hm1[key] = [0, 0]
    hm1[key][0] += math.log10(budget) if budget > 0 else 0
    hm1[key][1] += 1

z1 = [[round(hm1[(vi, pi)][0] / hm1[(vi, pi)][1], 2) if (vi, pi) in hm1 else None for pi in range(n_bins)] for vi in range(n_bins)]

fig = go.Figure(data=go.Heatmap(
    z=z1, x=pop_labels, y=vote_labels,
    colorscale=[[0, '#EEEEF3'], [0.5, '#4A7FD4'], [1, '#1a1a2e']],
    hoverongaps=False,
    hovertemplate='Popularity: %{x}<br>Vote Avg: %{y}<br>Avg Budget: %{z} (log₁₀)<extra></extra>',
    colorbar=dict(title='Budget (log₁₀)', tickvals=[4, 5, 6, 7, 8], ticktext=['$10K', '$100K', '$1M', '$10M', '$100M'])
))
fig.update_layout(
    title='Budget by Popularity × Vote Average',
    xaxis_title='Popularity', yaxis_title='Vote Average',
    width=900, height=500
)
fig.show()

In [17]:
# Heatmap 2: Popularity (X) × Budget (Y) → Avg Vote Average color
hm2 = {}
for pop, vote, budget, revenue in hm_df:
    pi, bi = find_bin(pop, pop_edges), find_bin(budget, budget_edges)
    key = (bi, pi)
    if key not in hm2:
        hm2[key] = [0, 0]
    hm2[key][0] += vote
    hm2[key][1] += 1

z2 = [[round(hm2[(bi, pi)][0] / hm2[(bi, pi)][1], 1) if (bi, pi) in hm2 else None for pi in range(n_bins)] for bi in range(n_bins)]

fig = go.Figure(data=go.Heatmap(
    z=z2, x=pop_labels, y=budget_labels,
    colorscale=[[0, '#EEEEF3'], [0.5, '#e8a838'], [1, '#c0392b']],
    hoverongaps=False,
    hovertemplate='Popularity: %{x}<br>Budget: %{y}<br>Avg Vote: %{z:.1f}<extra></extra>',
    colorbar=dict(title='Avg Vote')
))
fig.update_layout(
    title='Vote Average by Popularity × Budget',
    xaxis_title='Popularity', yaxis_title='Budget',
    width=900, height=500
)
fig.show()

In [18]:
# Heatmap 3: Popularity (X) × Vote Average (Y) → Avg log₁₀(Revenue) color
hm3 = {}
for pop, vote, budget, revenue in hm_df:
    pi, vi = find_bin(pop, pop_edges), find_bin(vote, vote_edges)
    key = (vi, pi)
    if key not in hm3:
        hm3[key] = [0, 0]
    hm3[key][0] += math.log10(revenue) if revenue > 0 else 0
    hm3[key][1] += 1

z3 = [[round(hm3[(vi, pi)][0] / hm3[(vi, pi)][1], 2) if (vi, pi) in hm3 else None for pi in range(n_bins)] for vi in range(n_bins)]
z3_flat = [v for row in z3 for v in row if v is not None]

fig = go.Figure(data=go.Heatmap(
    z=z3, x=pop_labels, y=vote_labels,
    zmin=min(z3_flat), zmax=max(z3_flat),
    colorscale=[[0, '#e8f5e9'], [0.25, '#81c784'], [0.5, '#27ae60'], [0.75, '#1b7a3d'], [1, '#0d3318']],
    hoverongaps=False,
    hovertemplate='Popularity: %{x}<br>Vote Avg: %{y}<br>Avg Revenue: %{z} (log₁₀)<extra></extra>',
    colorbar=dict(title='Revenue (log₁₀)')
))
fig.update_layout(
    title='Revenue by Popularity × Vote Average',
    xaxis_title='Popularity', yaxis_title='Vote Average',
    width=900, height=500
)
fig.show()